In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
import joblib
from time import time

# CatBoost import (raise clear error if missing)
try:
    from catboost import CatBoostClassifier
except Exception as e:
    raise ImportError('catboost is required: '+str(e)+" — run the install cell then restart the kernel")

# Locate dataset (robust search through parents)
project_root = Path.cwd()
data_path = None
for parent in [project_root] + list(project_root.parents):
    candidate = parent / 'Data' / 'Numerical' / 'Data' / '65_Nutrients_Data.csv'
    if candidate.exists():
        data_path = candidate
        project_root = parent
        break

if data_path is None:
    raise FileNotFoundError('Could not find 65_Nutrients_Data.csv under any parent of '+str(Path.cwd()))

print('Using data file:', data_path)
df = pd.read_csv(data_path)
print('df shape:', df.shape)
if 'novaclass' not in df.columns:
    raise KeyError('novaclass column missing in dataset')
X = df.drop(columns=['novaclass'])
y = df['novaclass'].astype(int)
min_label = int(y.min())
if min_label != 0:
    y = y - min_label
    print('Shifted labels by', min_label)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('train/test sizes', X_train.shape, X_test.shape)

# cross-validator
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# pipeline and search
sm = SMOTE(random_state=42)
cat = CatBoostClassifier(random_seed=42, verbose=0)
pipe_cat = Pipeline([('smote', sm), ('cat', cat)])
param_dist_cat = {
    'cat__iterations': [200, 500],
    'cat__depth': [6, 8],
    'cat__learning_rate': [0.03, 0.1],
}
rs_cat = RandomizedSearchCV(pipe_cat, param_distributions=param_dist_cat, n_iter=6, scoring='f1_macro', cv=cv, n_jobs=-1, random_state=42, verbose=1, return_train_score=True)
print('Starting CatBoost RandomizedSearchCV (6 trials)')
t0 = time()
rs_cat.fit(X_train, y_train)
t1 = time()
print('Done. Best CV f1_macro (CatBoost):', rs_cat.best_score_)
print('Best params (CatBoost):', rs_cat.best_params_)
best_cat = rs_cat.best_estimator_
y_pred_cat = best_cat.predict(X_test)
print('Test f1_macro (CatBoost):', f1_score(y_test, y_pred_cat, average='macro'))
print('Search time (s):', round(t1-t0,2))
out_dir_cat = project_root / 'models' / 'numerical_65_catboost' / 'smote'
out_dir_cat.mkdir(parents=True, exist_ok=True)
joblib.dump(best_cat, out_dir_cat / 'catboost_65_numerical_smote.joblib')
print('Saved CatBoost model to', out_dir_cat)